# Underwater Waste Detection — Dataset Exploration

This notebook walks through downloading and understanding the **TrashCan 1.0** dataset,
the primary benchmark for underwater trash detection.

**What we cover:**
- Mounting Google Drive
- Downloading TrashCan 1.0 (or loading from Drive if already there)
- Understanding the annotation format (COCO JSON → YOLO conversion preview)
- Class distribution analysis
- Sample image visualization
- Overview of other available public datasets in this space

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
!pip install -q pycocotools ultralytics opencv-python matplotlib seaborn pandas

In [ ]:
# ── Cell 2: Mount Google Drive ────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Set your Drive path here — all data and model weights will be stored here
DRIVE_BASE = '/content/drive/MyDrive/underwater_waste_detection'

import os
os.makedirs(DRIVE_BASE, exist_ok=True)
print(f'Drive base directory: {DRIVE_BASE}')

In [ ]:
# ── Cell 3: Clone project repo ────────────────────────────────────────────
# Replace with your actual GitHub repo URL after pushing
REPO_URL = 'https://github.com/omprxkash/underwater-waste-detection'
REPO_DIR = '/content/underwater-waste-detection'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# ── Cell 4: Download TrashCan 1.0 ─────────────────────────────────────────
# TrashCan 1.0 — University of Minnesota Data Repository
# DOI: https://doi.org/10.13020/g1yz-6p51
# License: CC BY 4.0

import zipfile
from pathlib import Path

DATASET_DIR = Path(DRIVE_BASE) / 'trashcan_raw'
DATASET_DIR.mkdir(parents=True, exist_ok=True)

ZIP_PATH = DATASET_DIR / 'trashcan_v1.zip'

if not ZIP_PATH.exists():
    print('Downloading TrashCan 1.0 (~2GB)...')
    DOWNLOAD_URL = (
        'https://conservancy.umn.edu/bitstream/handle/11299/214366/'
        'material_version1.zip?sequence=11&isAllowed=y'
    )
    !wget -q --show-progress "{DOWNLOAD_URL}" -O "{ZIP_PATH}"
    print('Download complete.')
else:
    print(f'Archive already exists: {ZIP_PATH}')

EXTRACT_DIR = DATASET_DIR / 'extracted'
if not EXTRACT_DIR.exists():
    print('Extracting...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
    print('Extraction complete.')

# Show what was extracted
print('\nExtracted contents:')
for item in sorted(EXTRACT_DIR.iterdir()):
    print(f'  {item.name}')

In [ ]:
# ── Cell 5: Load and inspect COCO annotations ─────────────────────────────
import json
import numpy as np

# Find annotation JSON files (TrashCan has both 'material' and 'instance' annotations)
ann_files = list(EXTRACT_DIR.rglob('*.json'))
print(f'Found {len(ann_files)} annotation files:')
for f in ann_files:
    print(f'  {f.relative_to(EXTRACT_DIR)}')

# Load the training annotation file (material annotations = coarser, good baseline)
# Adjust path based on what was extracted
train_ann_path = next((f for f in ann_files if 'train' in f.name.lower()), ann_files[0])
print(f'\nLoading: {train_ann_path.name}')

with open(train_ann_path) as f:
    coco_data = json.load(f)

print(f"Images: {len(coco_data['images'])}")
print(f"Annotations: {len(coco_data['annotations'])}")
print(f"Categories: {len(coco_data['categories'])}")
print('\nAll categories:')
for cat in coco_data['categories']:
    print(f"  {cat['id']:3d}: {cat['name']}")

In [ ]:
# ── Cell 6: Class distribution ────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

# Map TrashCan classes → macro-categories
TRASH_CLASSES = {
    'plastic_bag', 'plastic_bottle', 'milk_jug', 'paper_bag', 'cup',
    'container', 'bottle_cap', 'net', 'pipe', 'plastic_tray',
    'rubber_glove', 'rubber_boot', 'styrofoam', 'unknown_instance', 'unlabeled_trash'
}
BIO_CLASSES = {
    'fish', 'eel', 'crab', 'starfish', 'nudibranch', 'coral',
    'shell', 'lobster', 'octopus', 'sea_cucumber', 'shrimp', 'jellyfish'
}

cat_id_to_name = {cat['id']: cat['name'] for cat in coco_data['categories']}

# Count annotations per fine-grained class
class_counts = {}
for ann in coco_data['annotations']:
    name = cat_id_to_name.get(ann['category_id'], 'unknown')
    class_counts[name] = class_counts.get(name, 0) + 1

# Count macro-categories
macro_counts = {'trash': 0, 'bio': 0, 'rov': 0}
for name, count in class_counts.items():
    if name in TRASH_CLASSES:
        macro_counts['trash'] += count
    elif name in BIO_CLASSES:
        macro_counts['bio'] += count
    else:
        macro_counts['rov'] += count

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Fine-grained class bar chart
sorted_classes = sorted(class_counts.items(), key=lambda x: -x[1])
names_sorted = [c[0] for c in sorted_classes]
counts_sorted = [c[1] for c in sorted_classes]

colors = ['#e74c3c' if n in TRASH_CLASSES else '#27ae60' if n in BIO_CLASSES else '#3498db'
          for n in names_sorted]
ax1.barh(names_sorted, counts_sorted, color=colors)
ax1.set_xlabel('Number of Annotations')
ax1.set_title('Fine-grained Class Distribution (TrashCan 1.0)', fontweight='bold')
ax1.invert_yaxis()

# Macro-category pie chart
ax2.pie(
    macro_counts.values(),
    labels=[f"{k}\n({v:,})" for k, v in macro_counts.items()],
    colors=['#e74c3c', '#27ae60', '#3498db'],
    autopct='%1.1f%%',
    startangle=90,
    textprops={'fontsize': 12}
)
ax2.set_title('Macro-category Distribution (our mapping)', fontweight='bold')

plt.suptitle('TrashCan 1.0 — Annotation Distribution', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/class_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

print(f"\nMacro-category totals:")
for k, v in macro_counts.items():
    print(f"  {k:<10} {v:,} annotations")

In [ ]:
# ── Cell 7: Visualize sample images ───────────────────────────────────────
import cv2
from PIL import Image
import random

# Find image directory
img_dirs = list(EXTRACT_DIR.rglob('images'))
if img_dirs:
    image_dir = img_dirs[0]
    all_images = list(image_dir.rglob('*.jpg')) + list(image_dir.rglob('*.png'))
    print(f'Found {len(all_images)} images in {image_dir}')

    # Sample 8 random images
    sampled_imgs = random.sample(all_images, min(8, len(all_images)))

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()

    for idx, img_path in enumerate(sampled_imgs):
        img = Image.open(img_path)
        axes[idx].imshow(img)
        axes[idx].axis('off')
        w, h = img.size
        axes[idx].set_title(f'{img_path.name[:20]}\n{w}x{h}', fontsize=8)

    plt.suptitle('Sample Images from TrashCan 1.0', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Image directory not found. Check extracted folder structure.')

In [ ]:
# ── Cell 8: Image size statistics ─────────────────────────────────────────
import pandas as pd

widths = [img['width'] for img in coco_data['images']]
heights = [img['height'] for img in coco_data['images']]

size_df = pd.DataFrame({'width': widths, 'height': heights})
print('Image size statistics:')
print(size_df.describe().round(1))
print(f'\nMost common size: {size_df.mode().iloc[0]["width"]:.0f}x{size_df.mode().iloc[0]["height"]:.0f}')

# Aspect ratio distribution
aspect_ratios = [w/h for w, h in zip(widths, heights)]
print(f'Aspect ratio mean: {np.mean(aspect_ratios):.2f} (std: {np.std(aspect_ratios):.2f})')

## Other Public Datasets for Underwater Waste Detection

Below is a curated list of all public datasets available in this domain. We use **TrashCan 1.0** as our primary dataset, but these can be used to augment training or for cross-dataset evaluation.

| Dataset | Images | Classes | Format | Source | Notes |
|---|---|---|---|---|---|
| **TrashCan 1.0** | ~7,000 | 16 (→ 3 macro) | COCO | UMN DRUM | Primary dataset, CC BY |
| **Trash-ICRA19** | ~5,700 | 6 | COCO | MBZIRC 2019 | Underwater robot pickup task |
| **RUWI** | ~2,000 | 4 | Various | Research | Real Underwater Waste Images |
| **SUIM** | ~1,500 | 8 | Pixel mask | U of Manitoba | Includes debris + bio classes |
| **TACO** | ~1,500 | 60 | COCO | Open | General litter, not underwater; useful for transfer |
| **DeepFish** | ~40,000 | 1 (fish) | Bbox | ARC | Useful for bio-class negative training |
| **Roboflow Universe** | Varies | Varies | YOLO | Roboflow | Several community underwater detection datasets |
| **URPC2020** | ~5,000 | 4 | XML | Competition | Underwater robot picking challenge |

**Dataset download sources:**
- TrashCan 1.0: https://conservancy.umn.edu/items/431d7b20-e2f9-4fea-9eca-1c2e7c03fcc4
- Trash-ICRA19: https://github.com/pedropro/TACO (and iEEE DataPort)
- SUIM: https://irvlab.cs.umn.edu/resources/suim-dataset
- TACO: https://github.com/pedropro/TACO
- Roboflow: https://universe.roboflow.com/ (search 'underwater trash')

In [ ]:
# ── Cell 9: Summary ───────────────────────────────────────────────────────
print('=== Dataset Exploration Summary ===')
print(f'Dataset: TrashCan 1.0')
print(f'Total images: {len(coco_data["images"]):,}')
print(f'Total annotations: {len(coco_data["annotations"]):,}')
print(f'Fine-grained classes: {len(coco_data["categories"])}')
print(f'Macro-categories (our mapping): 3 (trash / bio / rov)')
print()
print('Class imbalance note:')
total_anns = sum(macro_counts.values())
for k, v in macro_counts.items():
    print(f'  {k:<10}: {v:5,} ({v/total_anns*100:.1f}%)')
print()
print('Next step → Run 02_training_baseline.ipynb to convert and train')